# NeuralMath — A anatomia matemática de uma rede neural

Notebook complementar ao artigo de **Americo Cunha Jr**.

O objetivo é implementar uma pequena rede neural **do zero**, usando apenas NumPy, e tornar explícita a correspondência entre as equações matemáticas e o código.

A rede possui duas entradas (massa e diâmetro), três neurônios ocultos com ReLU e uma saída sigmoide. O treinamento usa erro quadrático, retropropagação e descida do gradiente estocástica.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def relu(s):
    return np.maximum(0, s)

def relu_derivada(s):
    return (s > 0).astype(float)

def sigmoid(s):
    return 1 / (1 + np.exp(-s))


## Dados

O conjunto foi construído deliberadamente com uma geometria do tipo XOR, de modo que nenhuma única reta consiga separar as duas classes.


In [ ]:
X_treino = np.array([
    [130.0, 6.6], [134.0, 6.5],
    [170.0, 7.8], [174.0, 7.7],
    [130.0, 7.8], [126.0, 7.7],
    [170.0, 6.6], [166.0, 6.5]
])

y_treino = np.array([1., 1., 1., 1., 0., 0., 0., 0.])

X_teste = np.array([
    [132.0, 6.55], [172.0, 7.75],
    [128.0, 7.75], [168.0, 6.55]
])

y_teste = np.array([1., 1., 0., 0.])

media = X_treino.mean(axis=0)
desvio = X_treino.std(axis=0)
Xn = (X_treino - media) / desvio
Xn_teste = (X_teste - media) / desvio


## Treinamento

As derivadas são escritas explicitamente. Assim, a retropropagação aparece no código como uma aplicação direta da regra da cadeia.


In [ ]:
rng = np.random.default_rng(42)
W1 = 0.1 * rng.standard_normal((3, 2))
b1 = np.zeros(3)
W2 = 0.1 * rng.standard_normal(3)
b2 = 0.0

eta = 0.1
n_epocas = 2000
historico = []

for epoca in range(n_epocas):
    ordem = rng.permutation(len(Xn))

    for i in ordem:
        x = Xn[i]
        alvo = y_treino[i]

        z1 = W1 @ x + b1
        h = relu(z1)
        z2 = W2 @ h + b2
        y_hat = sigmoid(z2)

        delta2 = (y_hat - alvo) * y_hat * (1 - y_hat)
        dW2 = delta2 * h
        db2 = delta2

        delta1 = (delta2 * W2) * relu_derivada(z1)
        dW1 = np.outer(delta1, x)
        db1 = delta1

        W1 -= eta * dW1
        b1 -= eta * db1
        W2 -= eta * dW2
        b2 -= eta * db2

    erros = []
    for x, alvo in zip(Xn, y_treino):
        h = relu(W1 @ x + b1)
        y_hat = sigmoid(W2 @ h + b2)
        erros.append(0.5 * (alvo - y_hat)**2)

    erro_medio = np.mean(erros)
    historico.append(erro_medio)

    if epoca % 200 == 0:
        print(epoca, erro_medio)

print(1999, historico[-1])


In [ ]:
def prever(X):
    saidas = []
    for x in X:
        h = relu(W1 @ x + b1)
        y_hat = sigmoid(W2 @ h + b2)
        saidas.append(y_hat)
    return np.array(saidas)

print('Treinamento:', prever(Xn))
print('Teste:', prever(Xn_teste))


## Regiões de decisão aprendidas

A curva de nível $\widehat{y}=0{,}5$ constitui a fronteira de decisão produzida pela rede.


In [ ]:
massas = np.linspace(120, 180, 350)
diametros = np.linspace(6.3, 8.0, 350)
M, D = np.meshgrid(massas, diametros)

grade = np.column_stack([M.ravel(), D.ravel()])
grade_n = (grade - media) / desvio
P = prever(grade_n).reshape(M.shape)

fig, ax = plt.subplots(figsize=(8.2, 5.7))
ax.contourf(M, D, P, levels=[0, 0.5, 1], alpha=0.25)
ax.contour(M, D, P, levels=[0.5], linewidths=2)

for classe, rotulo in [(1., 'Maçã — treino'), (0., 'Laranja — treino')]:
    mask = y_treino == classe
    ax.scatter(X_treino[mask,0], X_treino[mask,1], s=70, label=rotulo)

for classe, rotulo in [(1., 'Maçã — teste'), (0., 'Laranja — teste')]:
    mask = y_teste == classe
    ax.scatter(X_teste[mask,0], X_teste[mask,1], marker='x', s=85, linewidths=2, label=rotulo)

ax.set_xlabel('Massa (g)')
ax.set_ylabel('Diâmetro (cm)')
ax.set_title('Saída da rede neural após o treinamento')
ax.legend()
plt.show()
